#The Spotter Assessment

In [1]:
!pip install lightgbm scikit-learn pandas numpy matplotlib

In [2]:
import pandas as pd
import numpy as np

def preprocess_and_engineer(df, is_train=True, category_maps=None):
    df = df.copy()

    # Extract date features
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    # Distance ratios and signal interaction features
    dist = df['distance'].clip(lower=1.0)
    df['weight_per_mile'] = df['weight'] / dist
    df['market_quote_ratio'] = df['market_index'] * df['quote_signal']

    # Categorical encoding
    categorical_cols = ['pickup', 'delivery', 'equipment']
    if is_train:
        category_maps = {}
        for col in categorical_cols:
            df[col] = df[col].astype('category')
            category_maps[col] = df[col].cat.categories
    else:
        for col in categorical_cols:
            df[col] = pd.Categorical(df[col], categories=category_maps[col])

    return df, category_maps

FEATURE_COLS = [
    'pickup', 'delivery', 'pickup_lat', 'pickup_lon',
    'delivery_lat', 'delivery_lon', 'distance', 'equipment',
    'weight', 'market_index', 'quote_signal', 'month',
    'day_of_week', 'day_of_month', 'is_weekend',
    'weight_per_mile', 'market_quote_ratio'
]

In [3]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load training data
train_df = pd.read_csv('train-test.csv')
train_processed, category_maps = preprocess_and_engineer(train_df, is_train=True)

X = train_processed[FEATURE_COLS]
y = train_processed['posted_rate']

# 5-Fold Cross Validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(train_df))
models = []

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'random_state': 42,
    'verbose': -1
}

print("--- Starting Model Training ---")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    oof_predictions[val_idx] = model.predict(X_val)
    models.append(model)

overall_rmse = np.sqrt(mean_squared_error(y, oof_predictions))
overall_r2 = r2_score(y, oof_predictions)

print(f"\nTraining Complete! Overall RMSE: {overall_rmse:.2f} | R² Score: {overall_r2:.4f}")

--- Starting Model Training ---

Training Complete! Overall RMSE: 600.53 | R² Score: 0.8368


In [4]:
# 1. Validation predictions
val_df = pd.read_csv('validation.csv')
val_processed, _ = preprocess_and_engineer(val_df, is_train=False, category_maps=category_maps)
val_preds = np.mean([m.predict(val_processed[FEATURE_COLS]) for m in models], axis=0)

val_output = pd.DataFrame({
    'load_id': val_df['load_id'],
    'predicted_rate': np.round(val_preds, 2)
})
val_output.to_csv('validation_predictions.csv', index=False)
print(" Saved: validation_predictions.csv")

# 2. December trend predictions
dec_df = pd.read_csv('december-chart-inputs.csv')
dec_df['pickup_lat'] = train_df[train_df['pickup']=='Lexington']['pickup_lat'].median()
dec_df['pickup_lon'] = train_df[train_df['pickup']=='Lexington']['pickup_lon'].median()
dec_df['delivery_lat'] = train_df[train_df['delivery']=='Fort Wayne']['delivery_lat'].median()
dec_df['delivery_lon'] = train_df[train_df['delivery']=='Fort Wayne']['delivery_lon'].median()
dec_df['market_index'] = train_df['market_index'].median()
dec_df['quote_signal'] = train_df['quote_signal'].median()

dec_processed, _ = preprocess_and_engineer(dec_df, is_train=False, category_maps=category_maps)
dec_preds = np.mean([m.predict(dec_processed[FEATURE_COLS]) for m in models], axis=0)

dec_df_output = pd.read_csv('december-chart-inputs.csv')
dec_df_output['predicted_rate'] = np.round(dec_preds, 2)
dec_df_output.to_csv('december-chart-inputs.csv', index=False)
print(" Saved: december-chart-inputs.csv")

 Saved: validation_predictions.csv
 Saved: december-chart-inputs.csv


In [5]:
!python Score.py --predictions validation_predictions.csv --december-predictions december-chart-inputs.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
